# SunCoke Energy (NYSE: SXC) — 3-Statement & Valuation Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k-sifat/SXC-Financial-Model/blob/main/SXC_Historical_Model.ipynb)

## Executive Summary

This notebook replicates the full 3-statement financial model originally built in `SXC-historical-model.xlsx`. It projects SunCoke Energy's Income Statement, Balance Sheet, and Cash Flow Statement over a **5-year forecast horizon (FY2026E – FY2030E)** and derives two independent equity valuations:

| Method | Description |
|--------|-------------|
| **Discounted Cash Flow (DCF)** | Gordon Growth terminal value (g = 0%), discounted at WACC |
| **EV/EBITDA Multiple** | 3-year forward price target using FY28E EBITDA × target multiple, bridged to equity |

The **blended price target** is the simple average (50/50) of the two methods. A 3-year annualized IRR is computed from the blended target relative to the current share price.

### Key Model Features
- **Revenue Build**: Domestic coke volume × revenue-per-ton + IS segment (margin-implied) + Brazil/Other
- **Granite City Toggle**: Conditional 590 kt volume loss from FY27+ if the contract is not renewed
- **Debt Schedule**: $dollar40M/year revolver paydown + $500M Senior Notes (4.875% coupon, held flat)
- **PP&E Roll-Forward**: Recursive `PPE(t) = PPE(t-1) + CapEx(t) - D&A(t)` with D&A as % of prior-year net PP&E
- **Growth CapEx Wind-Down**: Phoenix integration spending tapers from &dollar20M → &dollar0M over FY26–FY30
- **Working Capital Delta**: Flows into unlevered FCF via `UFCF = NOPAT + D&A + ΔWC - CapEx`

> **Static vs. Interactive**: Cell 5 below shows the **base-case results pre-rendered** so GitHub visitors see the full output without running any code. Cell 6 provides an interactive slider dashboard for scenario analysis (requires a live kernel — use Colab or run locally).

## 1. Setup & Financial Constants

All constants below are sourced directly from the Historical Financials tab and Projections tab of `SXC-historical-model.xlsx`. They are **hardcoded intentionally** — the model's interactive drivers (Cell 6) adjust the projection mechanics, not the anchor data.

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ═══════════════════════════════════════════════════════════════════
# MARKET DATA — as of 8/13/26
# ═══════════════════════════════════════════════════════════════════

SHARE_PRICE        = 9.29        # Valuation B4 — as of 8/13/26
SHARES_OUT         = 85.5        # Historical C28 — millions

# ═══════════════════════════════════════════════════════════════════
# FY2025 ACTUALS (Historical Financials tab, column C)
# ═══════════════════════════════════════════════════════════════════

FY25_PPE           = 1202.7      # C43 — Net PP&E
FY25_CASH          = 88.7        # C37 — Cash
FY25_REVENUE       = 1837.3      # C7  — Total Revenue
FY25_EBITDA        = 259.7       # DCF F8 (= Historical E146 for FY25 actuals used by DCF)

# ═══════════════════════════════════════════════════════════════════
# DEBT SCHEDULE
# ═══════════════════════════════════════════════════════════════════

# Revolver
REVOLVER_OPEN_FY26 = 185.5       # Projections C16 — revolver closing FY25
REVOLVER_PAYDOWN   = 40.0        # $40M/year paydown
REVOLVER_RATE      = 0.06        # 6% SOFR+spread

# Senior Notes
SENIOR_NOTES_BAL   = 500.0       # $500M balance, held flat
SENIOR_NOTES_RATE  = 0.04875     # 4.875% coupon

# ═══════════════════════════════════════════════════════════════════
# REVENUE BUILD CONSTANTS
# ═══════════════════════════════════════════════════════════════════

DC_REV_PER_TON     = 430.46145638837027   # Projections C39 — FY25 actual, held flat
IS_EBITDA_MARGIN   = 0.332                # Projections C40 — 33.2%
BRAZIL_OTHER_REV   = 35.7                 # Projections D48 — held flat

# ═══════════════════════════════════════════════════════════════════
# EBITDA & OPERATING CONSTANTS
# ═══════════════════════════════════════════════════════════════════

CORP_OTHER_EBITDA  = -13.1       # Projections D37 — held flat

# CapEx schedule (FY26–FY30)
MAINT_CAPEX        = 50.0        # $50M/year maintenance
GROWTH_CAPEX       = [20.0, 15.0, 10.0, 5.0, 0.0]  # Phoenix integration wind-down

# ═══════════════════════════════════════════════════════════════════
# DIVIDENDS, NCI & CFS CONSTANTS
# ═══════════════════════════════════════════════════════════════════

DIVIDENDS          = 41.6        # $41.6M/year
NCI_NET_INCOME     = 5.4         # $5.4M/year
NCI_DISTRIBUTIONS  = 7.4         # $7.4M/year
SBC                = 3.0         # $3M/year share-based comp
OTHER_FINANCING    = -3.0        # $-3M/year lease repayments

# ═══════════════════════════════════════════════════════════════════
# BALANCE SHEET STATIC ITEMS (FY25 actuals)
# ═══════════════════════════════════════════════════════════════════

BS_RECEIVABLES     = 111.5
BS_INVENTORIES     = 219.9
BS_TAX_OTHER_CA    = 25.0        # tax receivable + other CA
BS_GOODWILL        = 55.6
BS_INTANGIBLES     = [40, 36, 32, 28, 24]  # amortising schedule FY26–FY30
BS_DEF_CHARGES     = 22.0
BS_AP              = 157.3
BS_ACCRUED_INT     = 60.0        # accrued liabilities + interest payable
BS_BLACK_LUNG      = 11.7
BS_RETIREMENT      = 7.0
BS_DEF_TAX         = 190.0
BS_ARO_OTHER_LT    = 47.0
BS_COMMON_STOCK    = 1.0
BS_TREASURY        = -184.0
BS_APIC            = 732.2
BS_AOCI            = -4.2
BS_RE_FY25         = 52.3        # Historical C71 — retained earnings FY25
BS_NCI_FY25        = 28.8        # Historical C72 — noncontrolling interests FY25

# ═══════════════════════════════════════════════════════════════════
# VALUATION CONSTANTS
# ═══════════════════════════════════════════════════════════════════

TERMINAL_GROWTH    = 0.0         # 0% long-term growth rate
GRANITE_CITY_LOSS  = 590         # kt lost if toggle = 0, from FY2027+

## 2. Valuation Engine — `run_model()`

The core financial engine projects 5 years (FY26–FY30) across three statements and derives two independent valuations. Every formula below mirrors the exact cell references in `SXC-historical-model.xlsx`.

**Schedule Mechanics:**
1. **Coke Volume**: FY26 uses full base; FY27+ applies the Granite City toggle (−590 kt if lost)
2. **Revolver**: Opens at prior close, pays down $40M/year, floors at $0; average = (open + close) / 2
3. **D&A**: `da_rate × prior-year net PP&E`; PP&E rolls forward recursively
4. **Working Capital**: Only cash varies; receivables, inventories, payables held flat from FY25
5. **UFCF**: `NOPAT + D&A + ΔWC − CapEx` (levered items excluded)
6. **Terminal Value**: Gordon Growth `TV = FCF₅ × (1+g) / (WACC−g)`, g = 0%
7. **EV/EBITDA Exit**: FY28 EBITDA × multiple, bridge to equity via FY28 net debt

In [2]:
def run_model(tax_rate_pct, da_pct, sga_pct, coke_vol_base, ebitda_per_ton,
              is_ebitda, granite_toggle, wacc_pct, target_multiple):
    """
    Runs the full 3-statement projection (FY26-FY30), DCF, and EV/EBITDA valuation.
    Returns a dict of results.
    """
    tax_rate = tax_rate_pct / 100.0    # Drivers E7 / 100
    da_rate  = da_pct / 100.0          # Drivers E8 / 100
    wacc     = wacc_pct / 100.0

    # ── PROJECTION ARRAYS — 5 years: FY26(0), FY27(1), FY28(2), FY29(3), FY30(4) ──

    # --- Industry Drivers ---
    # Coke Sales Volume — Applied (k tons)
    # FY26: always uses full base volume (Proj D33 = D31)
    # FY27+: IF(toggle=1, base, base-590)
    coke_applied = [0.0] * 5
    coke_applied[0] = coke_vol_base
    for i in range(1, 5):
        if granite_toggle == 1:
            coke_applied[i] = coke_vol_base
        else:
            coke_applied[i] = coke_vol_base - GRANITE_CITY_LOSS

    # Domestic Coke Adj. EBITDA ($M) = applied volume x EBITDA/ton / 1000
    dc_ebitda = [coke_applied[i] * ebitda_per_ton / 1000.0 for i in range(5)]

    # Industrial Services Adj. EBITDA ($M) — direct from driver
    is_ebitda_arr = [is_ebitda] * 5

    # Total Adj. EBITDA ($M) = DC + IS + Corp & Other
    total_ebitda = [dc_ebitda[i] + is_ebitda_arr[i] + CORP_OTHER_EBITDA for i in range(5)]

    # --- Revenue Build ---
    dc_revenue = [coke_applied[i] * DC_REV_PER_TON / 1000.0 for i in range(5)]
    is_revenue = [is_ebitda_arr[i] / IS_EBITDA_MARGIN for i in range(5)]
    total_revenue = [dc_revenue[i] + is_revenue[i] + BRAZIL_OTHER_REV for i in range(5)]

    # --- Debt & Interest Schedule ---
    revolver_open = [0.0] * 5
    revolver_close = [0.0] * 5
    revolver_avg = [0.0] * 5
    revolver_open[0] = REVOLVER_OPEN_FY26

    for i in range(5):
        if i > 0:
            revolver_open[i] = max(revolver_close[i - 1], 0.0)
        revolver_close[i] = max(revolver_open[i] - REVOLVER_PAYDOWN, 0.0)
        revolver_avg[i] = (revolver_open[i] + revolver_close[i]) / 2.0

    interest_senior = [SENIOR_NOTES_BAL * SENIOR_NOTES_RATE] * 5
    interest_revolver = [revolver_avg[i] * REVOLVER_RATE for i in range(5)]
    interest_total = [interest_senior[i] + interest_revolver[i] for i in range(5)]

    # --- CapEx Schedule ---
    total_capex = [MAINT_CAPEX + GROWTH_CAPEX[i] for i in range(5)]

    # --- Income Statement ---
    sga = [total_revenue[i] * sga_pct for i in range(5)]
    cogs = [total_revenue[i] - total_ebitda[i] - sga[i] for i in range(5)]

    # D&A: FY26 uses FY25 PP&E; FY27+ uses prior year projected PP&E
    ppe = [0.0] * 5
    da = [0.0] * 5
    da[0] = da_rate * FY25_PPE
    ppe[0] = FY25_PPE + total_capex[0] - da[0]
    for i in range(1, 5):
        da[i] = da_rate * ppe[i - 1]
        ppe[i] = ppe[i - 1] + total_capex[i] - da[i]

    total_opex = [cogs[i] + sga[i] + da[i] for i in range(5)]
    ebit = [total_revenue[i] - total_opex[i] for i in range(5)]
    ebt = [ebit[i] - interest_total[i] for i in range(5)]
    tax = [ebt[i] * tax_rate for i in range(5)]
    net_income_consol = [ebt[i] - tax[i] for i in range(5)]
    ni_to_sxc = [net_income_consol[i] - NCI_NET_INCOME for i in range(5)]

    # --- Balance Sheet ---
    retained_earnings = [0.0] * 5
    retained_earnings[0] = BS_RE_FY25 + ni_to_sxc[0] - DIVIDENDS
    for i in range(1, 5):
        retained_earnings[i] = retained_earnings[i - 1] + ni_to_sxc[i] - DIVIDENDS

    nci_equity = [0.0] * 5
    nci_equity[0] = BS_NCI_FY25 + NCI_NET_INCOME - NCI_DISTRIBUTIONS
    for i in range(1, 5):
        nci_equity[i] = nci_equity[i - 1] + NCI_NET_INCOME - NCI_DISTRIBUTIONS

    lt_debt = [SENIOR_NOTES_BAL + revolver_close[i] for i in range(5)]

    # --- Cash Flow Statement ---
    cfo = [net_income_consol[i] + da[i] + SBC for i in range(5)]
    cfi = [-total_capex[i] for i in range(5)]

    cff = [0.0] * 5
    for i in range(5):
        rev_repay = -(revolver_open[i] - revolver_close[i])
        cff[i] = rev_repay + (-DIVIDENDS) + (-NCI_DISTRIBUTIONS) + OTHER_FINANCING

    net_cash_change = [cfo[i] + cfi[i] + cff[i] for i in range(5)]

    cash = [0.0] * 5
    cash[0] = FY25_CASH + net_cash_change[0]
    for i in range(1, 5):
        cash[i] = cash[i - 1] + net_cash_change[i]

    # --- Working Capital (for DCF NWC delta) ---
    fy25_total_ca = FY25_CASH + BS_RECEIVABLES + BS_INVENTORIES + BS_TAX_OTHER_CA
    fy25_total_cl = BS_AP + BS_ACCRUED_INT
    fy25_wc = fy25_total_ca - fy25_total_cl

    wc = [0.0] * 5
    for i in range(5):
        total_ca = cash[i] + BS_RECEIVABLES + BS_INVENTORIES + BS_TAX_OTHER_CA
        total_cl = BS_AP + BS_ACCRUED_INT
        wc[i] = total_ca - total_cl

    delta_wc = [0.0] * 5
    delta_wc[0] = wc[0] - fy25_wc
    for i in range(1, 5):
        delta_wc[i] = wc[i] - wc[i - 1]

    # ── DCF VALUATION ──
    nopat = [ebit[i] * (1 - tax_rate) for i in range(5)]
    ufcf = [nopat[i] + da[i] + delta_wc[i] + (-total_capex[i]) for i in range(5)]
    discount_factors = [1.0 / (1 + wacc) ** (i + 1) for i in range(5)]
    pv_fcf = [ufcf[i] * discount_factors[i] for i in range(5)]

    terminal_fcf = ufcf[4]
    if wacc - TERMINAL_GROWTH > 0:
        terminal_value = terminal_fcf * (1 + TERMINAL_GROWTH) / (wacc - TERMINAL_GROWTH)
    else:
        terminal_value = 0.0

    pv_terminal = terminal_value * discount_factors[4]
    pv_forecast = sum(pv_fcf)
    enterprise_value_dcf = pv_terminal + pv_forecast

    dcf_cash = cash[4]
    dcf_total_debt = lt_debt[4]
    dcf_net_debt = dcf_total_debt - dcf_cash
    dcf_equity = enterprise_value_dcf - dcf_net_debt
    dcf_price = dcf_equity / SHARES_OUT

    # ── EV/EBITDA VALUATION (3-Year Price Target) ──
    fy28_ebitda = total_ebitda[2]
    ev_ebitda = fy28_ebitda * target_multiple
    fy28_net_debt = lt_debt[2] - cash[2]
    mkt_cap_3yr = ev_ebitda - fy28_net_debt
    ev_ebitda_pt = mkt_cap_3yr / SHARES_OUT

    # ── BLENDED PRICE TARGET ──
    blended_pt = (dcf_price + ev_ebitda_pt) / 2.0

    # ── KEY RATIOS ──
    if SHARE_PRICE > 0 and blended_pt > 0:
        irr_3yr = (blended_pt / SHARE_PRICE) ** (1.0 / 3.0) - 1.0
    else:
        irr_3yr = 0.0

    rev_growth = (total_revenue[0] - FY25_REVENUE) / abs(FY25_REVENUE)

    mkt_cap_today = SHARE_PRICE * SHARES_OUT
    tev_today = mkt_cap_today + (SENIOR_NOTES_BAL + REVOLVER_OPEN_FY26) - FY25_CASH

    ev_ebitda_exit = ev_ebitda / fy28_ebitda if fy28_ebitda != 0 else 0
    ev_ebitda_3yr = tev_today / fy28_ebitda if fy28_ebitda != 0 else 0
    ev_ebitda_now = tev_today / FY25_EBITDA if FY25_EBITDA != 0 else 0

    return {
        'ev_ebitda_pt': ev_ebitda_pt, 'dcf_pt': dcf_price, 'blended_pt': blended_pt,
        'irr_3yr': irr_3yr, 'rev_growth': rev_growth, 'tev_today': tev_today,
        'ev_ebitda_exit': ev_ebitda_exit, 'ev_ebitda_3yr': ev_ebitda_3yr, 'ev_ebitda_now': ev_ebitda_now,
        'fy28_ebitda': fy28_ebitda, 'fy28_net_debt': fy28_net_debt,
        'total_revenue': total_revenue, 'total_ebitda': total_ebitda,
        'net_income_consol': net_income_consol, 'ni_to_sxc': ni_to_sxc,
        'ebit': ebit, 'cash': cash, 'lt_debt': lt_debt, 'ppe': ppe, 'da': da,
        'ufcf': ufcf, 'pv_fcf': pv_fcf, 'pv_terminal': pv_terminal,
        'pv_forecast': pv_forecast, 'enterprise_value_dcf': enterprise_value_dcf,
        'terminal_value': terminal_value, 'cfo': cfo, 'cfi': cfi, 'cff': cff,
    }

## 3. Output Presentation — `format_output()`

Generates a styled HTML report with:
- **Price target cards** (EV/EBITDA, DCF, Blended) with upside % and 3-year IRR
- **Key ratios table** (IRR, revenue growth, TEV, EV/EBITDA multiples)
- **5-year projection summary** (Revenue, EBITDA, EBIT, NI, D&A, PP&E, Cash, Debt, CFO, UFCF)
- **DCF breakdown** (PV forecast, terminal value, enterprise value, price per share)
- **EV/EBITDA bridge** (FY28 EBITDA → EV → equity → price target)

In [3]:
def format_output(r):
    """Build an HTML string displaying price targets and ratios."""
    years = ['FY26E', 'FY27E', 'FY28E', 'FY29E', 'FY30E']

    def pct(v):
        return f"{v * 100:.2f}%"

    def dollar(v):
        return f"${v:,.2f}"

    def money(v):
        return f"${v:,.1f}M"

    html = """
    <style>
        .sxc-box { font-family: 'Segoe UI', Calibri, sans-serif; max-width: 780px; }
        .sxc-box h2 { margin: 18px 0 8px 0; color: #1a3a5c; border-bottom: 2px solid #1a3a5c; padding-bottom: 4px; font-size: 16px; }
        .sxc-box h3 { margin: 14px 0 6px 0; color: #2d6a9f; font-size: 14px; }
        .sxc-box table { border-collapse: collapse; width: 100%; margin-bottom: 10px; font-size: 13px; }
        .sxc-box th, .sxc-box td { padding: 5px 10px; text-align: right; border-bottom: 1px solid #e0e0e0; }
        .sxc-box th { background: #f0f4f8; color: #1a3a5c; font-weight: 600; }
        .sxc-box td:first-child, .sxc-box th:first-child { text-align: left; }
        .sxc-pt { font-size: 22px; font-weight: 700; color: #0d6e3f; }
        .sxc-neg { color: #c0392b; }
        .sxc-card { display: inline-block; background: #f8fafe; border: 1px solid #d0dbe8;
                     border-radius: 8px; padding: 12px 20px; margin: 6px 8px 6px 0; min-width: 200px; text-align: center; }
        .sxc-card-label { font-size: 11px; color: #5a7a9a; text-transform: uppercase; letter-spacing: 0.5px; }
        .sxc-card-value { font-size: 20px; font-weight: 700; margin-top: 4px; }
        .sxc-green { color: #0d6e3f; }
        .sxc-blue { color: #1a3a5c; }
    </style>
    <div class="sxc-box">
    """

    # Price Target Cards
    upside_ev = (r['ev_ebitda_pt'] / SHARE_PRICE - 1)
    upside_dcf = (r['dcf_pt'] / SHARE_PRICE - 1)
    upside_blend = (r['blended_pt'] / SHARE_PRICE - 1)
    irr_ev = ((r['ev_ebitda_pt'] / SHARE_PRICE) ** (1/3) - 1) if r['ev_ebitda_pt'] > 0 else 0
    irr_dcf = ((r['dcf_pt'] / SHARE_PRICE) ** (1/3) - 1) if r['dcf_pt'] > 0 else 0

    def color_val(v, fmt="dollar"):
        cls = "sxc-green" if v >= 0 else "sxc-neg"
        if fmt == "dollar":
            txt = f"${v:,.2f}"
        else:
            txt = f"{v*100:,.1f}%"
        return f'<span class="{cls}">{txt}</span>'

    html += '<h2>Price Targets</h2>'
    html += '<div>'
    for label, pt_val, up_val, irr_val in [
        ("EV/EBITDA (3-Yr PT)", r['ev_ebitda_pt'], upside_ev, irr_ev),
        ("DCF Intrinsic Value", r['dcf_pt'], upside_dcf, irr_dcf),
        ("Blended PT", r['blended_pt'], upside_blend, r['irr_3yr']),
    ]:
        html += f'''
        <div class="sxc-card">
            <div class="sxc-card-label">{label}</div>
            <div class="sxc-card-value sxc-blue">{dollar(pt_val)}</div>
            <div style="font-size:11px; margin-top:3px;">
                Upside: {color_val(up_val, "pct")} &nbsp;|&nbsp; IRR: {color_val(irr_val, "pct")}
            </div>
        </div>'''
    html += f'<div style="font-size:12px; color:#888; margin-top:4px;">Current Price: ${SHARE_PRICE:.2f} &nbsp;|&nbsp; Shares Out: {SHARES_OUT}M</div>'
    html += '</div>'

    # Key Ratios
    html += '<h2>Key Ratios</h2>'
    html += '<table>'
    html += '<tr><th style="width:260px">Metric</th><th>Value</th></tr>'
    ratios = [
        ("3-Year IRR (Blended)", pct(r['irr_3yr'])),
        ("Revenue Growth (FY25 to FY26)", pct(r['rev_growth'])),
        ("TEV (Today)", money(r['tev_today'])),
        ("EV/EBITDA on Exit (FY28)", f"{r['ev_ebitda_exit']:.2f}x"),
        ("EV/EBITDA 3-Year (TEV / FY28 EBITDA)", f"{r['ev_ebitda_3yr']:.2f}x"),
        ("EV/EBITDA Today (TEV / FY25 EBITDA)", f"{r['ev_ebitda_now']:.2f}x"),
    ]
    for name, val in ratios:
        html += f'<tr><td>{name}</td><td>{val}</td></tr>'
    html += '</table>'

    # Projection Summary Table
    html += '<h2>Projection Summary (FY26E - FY30E)</h2>'
    html += '<table><tr><th style="width:200px">Line Item</th>'
    for y in years:
        html += f'<th>{y}</th>'
    html += '</tr>'

    rows_data = [
        ("Total Revenue", r['total_revenue'], money),
        ("Total Adj. EBITDA", r['total_ebitda'], money),
        ("EBIT", r['ebit'], money),
        ("Net Income (Consol.)", r['net_income_consol'], money),
        ("NI to SXC", r['ni_to_sxc'], money),
        ("D&A", r['da'], money),
        ("PP&E (net)", r['ppe'], money),
        ("Cash", r['cash'], money),
        ("LT Debt", r['lt_debt'], money),
        ("CFO", r['cfo'], money),
        ("Unlevered FCF (DCF)", r['ufcf'], money),
    ]
    for name, arr, fmt in rows_data:
        html += f'<tr><td>{name}</td>'
        for v in arr:
            cls = ' class="sxc-neg"' if v < 0 else ''
            html += f'<td{cls}>{fmt(v)}</td>'
        html += '</tr>'
    html += '</table>'

    # DCF Breakdown
    html += '<h3>DCF Breakdown</h3>'
    html += '<table>'
    dcf_items = [
        ("PV of Forecast Period CFs", money(r['pv_forecast'])),
        ("Terminal Value (undiscounted)", money(r['terminal_value'])),
        ("PV of Terminal Value", money(r['pv_terminal'])),
        ("Enterprise Value (DCF)", money(r['enterprise_value_dcf'])),
        ("DCF Price per Share", dollar(r['dcf_pt'])),
    ]
    for name, val in dcf_items:
        html += f'<tr><td>{name}</td><td>{val}</td></tr>'
    html += '</table>'

    # EV/EBITDA Breakdown
    html += '<h3>EV/EBITDA Breakdown</h3>'
    html += '<table>'
    ev_items = [
        ("FY28 Adj. EBITDA", money(r['fy28_ebitda'])),
        ("x Target Multiple", f"{r['ev_ebitda_exit']:.1f}x"),
        ("= Enterprise Value", money(r['fy28_ebitda'] * r['ev_ebitda_exit'])),
        ("- FY28 Net Debt", money(r['fy28_net_debt'])),
        ("= Equity Value", money(r['fy28_ebitda'] * r['ev_ebitda_exit'] - r['fy28_net_debt'])),
        ("/ Shares Out", f"{SHARES_OUT}M"),
        ("= 3-Year Price Target", dollar(r['ev_ebitda_pt'])),
    ]
    for name, val in ev_items:
        html += f'<tr><td>{name}</td><td>{val}</td></tr>'
    html += '</table>'

    html += '</div>'
    return html

## 4. Base-Case Static Output

The cell below runs the model with **base-case assumptions** and renders the results as static HTML. This output is saved in the notebook file so it displays on **GitHub without execution**.

| Driver | Base-Case Value |
|--------|----------------|
| Effective Tax Rate | 25.35% |
| D&A as % of Net PP&E | 12.5% |
| SG&A as % of Revenue | 4.5% |
| Coke Sales Volume | 3,300 kt |
| EBITDA per Ton (DC) | $48.00 |
| IS Segment EBITDA | $95.0M |
| Granite City Toggle | 0 (Contract Lost) |
| WACC | 9.38% |
| Target EV/EBITDA Multiple | 7.0x |

In [4]:
# Base-case model run — output is pre-rendered for GitHub static viewing
results = run_model(
    tax_rate_pct   = 25.35,
    da_pct         = 12.5,
    sga_pct        = 0.045,
    coke_vol_base  = 3300,
    ebitda_per_ton = 48.0,
    is_ebitda      = 95.0,
    granite_toggle = 0,
    wacc_pct       = 9.38,
    target_multiple= 7.0,
)
display(HTML(format_output(results)))

## 5. Interactive Scenario Sandbox

> **⚠️ Requires a live kernel.** The sliders below use `ipywidgets` and will **not render on GitHub's static viewer**. To use the interactive dashboard:
> - Click the **"Open in Colab"** badge at the top of this notebook, or
> - Clone the repo and run locally with `jupyter notebook SXC_Historical_Model.ipynb`

Adjust any of the 9 driver inputs and click **Run Model** to see updated price targets, key ratios, and financial projections.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INTERACTIVE WIDGETS — 9 adjustable inputs
# ═══════════════════════════════════════════════════════════════════

style = {'description_width': '220px'}
layout = widgets.Layout(width='520px')

w_tax_rate = widgets.FloatSlider(
    value=25.35, min=10.0, max=45.0, step=0.05,
    description='Effective Tax Rate (%):',
    style=style, layout=layout, readout_format='.2f'
)
w_da_pct = widgets.FloatSlider(
    value=12.5, min=5.0, max=25.0, step=0.1,
    description='D&A as % of Net PP&E:',
    style=style, layout=layout, readout_format='.1f'
)
w_sga_pct = widgets.FloatSlider(
    value=0.045, min=0.020, max=0.100, step=0.001,
    description='SG&A as % of Revenue:',
    style=style, layout=layout, readout_format='.3f'
)
w_coke_vol = widgets.FloatSlider(
    value=3300, min=2000, max=4500, step=50,
    description='Coke Sales Volume (k tons):',
    style=style, layout=layout, readout_format='.0f'
)
w_ebitda_ton = widgets.FloatSlider(
    value=48.0, min=30.0, max=70.0, step=0.5,
    description='EBITDA per Ton — DC ($/ton):',
    style=style, layout=layout, readout_format='.1f'
)
w_is_ebitda = widgets.FloatSlider(
    value=95.0, min=50.0, max=150.0, step=1.0,
    description='IS Segment EBITDA ($M):',
    style=style, layout=layout, readout_format='.1f'
)
w_granite = widgets.Dropdown(
    options=[('0 — Contract Lost (Bear)', 0), ('1 — Renews (Base)', 1)],
    value=0,
    description='Granite City Toggle:',
    style=style, layout=layout
)
w_wacc = widgets.FloatSlider(
    value=9.38, min=5.0, max=18.0, step=0.01,
    description='WACC (%):',
    style=style, layout=layout, readout_format='.2f'
)
w_multiple = widgets.FloatSlider(
    value=7.0, min=3.0, max=15.0, step=0.1,
    description='Target EV/EBITDA Multiple:',
    style=style, layout=layout, readout_format='.1f'
)

run_button = widgets.Button(description='Run Model', button_style='success',
                            layout=widgets.Layout(width='160px', height='40px'))
output = widgets.Output()


# ═══════════════════════════════════════════════════════════════════
# CALLBACK — wire up the button
# ═══════════════════════════════════════════════════════════════════

def on_run_click(_):
    with output:
        clear_output(wait=True)
        results = run_model(
            tax_rate_pct   = w_tax_rate.value,
            da_pct         = w_da_pct.value,
            sga_pct        = w_sga_pct.value,
            coke_vol_base  = w_coke_vol.value,
            ebitda_per_ton = w_ebitda_ton.value,
            is_ebitda      = w_is_ebitda.value,
            granite_toggle = w_granite.value,
            wacc_pct       = w_wacc.value,
            target_multiple= w_multiple.value,
        )
        display(HTML(format_output(results)))

run_button.on_click(on_run_click)


# ═══════════════════════════════════════════════════════════════════
# LAYOUT — display everything
# ═══════════════════════════════════════════════════════════════════

header = widgets.HTML(
    '<h1 style="font-family: Segoe UI, Calibri, sans-serif; color: #1a3a5c; margin-bottom: 0;">'
    'SunCoke Energy (SXC) &mdash; Interactive Valuation Model</h1>'
    '<p style="font-family: Segoe UI, sans-serif; color: #5a7a9a; margin-top: 4px;">'
    'Replicates <code>SXC-historical-model.xlsx</code> &nbsp;|&nbsp; Adjust drivers below and click <b>Run Model</b></p>'
)

driver_box = widgets.VBox([
    widgets.HTML('<b style="color:#1a3a5c;">Universal Drivers</b>'),
    w_tax_rate, w_da_pct, w_sga_pct,
    widgets.HTML('<b style="color:#1a3a5c; margin-top:8px;">Industry-Specific Drivers</b>'),
    w_coke_vol, w_ebitda_ton, w_is_ebitda, w_granite,
    widgets.HTML('<b style="color:#1a3a5c; margin-top:8px;">Valuation Assumptions</b>'),
    w_wacc, w_multiple,
])

display(header)
display(driver_box)
display(run_button)
display(output)

# Auto-run on first load
on_run_click(None)